In [ ]:
import kagglehub

import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

import pandas as pd
import os
from PIL import Image

import torch
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

from tqdm import tqdm
from transformers import AutoImageProcessor, SwinForImageClassification

## Prepare data

In [ ]:
path = kagglehub.dataset_download("kuanghueilee/food-101n")

print("Path to dataset files:", path)

In [ ]:
def check_path_exists(row, root_path):
    full_path = os.path.join(root_path, row['image_path'])
    return os.path.exists(full_path)

def add_noise_to_labels(df, noise_ratio=0.4):
    df_copy = df.copy()

    unique_labels = df_copy['food_label'].unique()

    n_samples = len(df_copy)
    n_noisy = int(n_samples * noise_ratio)
    noisy_indices = np.random.choice(n_samples, n_noisy, replace=False)

    for idx in noisy_indices:
        current_label = df_copy.loc[idx, 'food_label']
        possible_labels = [label for label in unique_labels if label != current_label]
        if possible_labels:
            new_label = np.random.choice(possible_labels)
            df_copy.loc[idx, 'noisy_food_label'] = int(new_label)

    non_noisy_indices = [i for i in range(n_samples) if i not in noisy_indices]
    df_copy.loc[non_noisy_indices, 'noisy_food_label'] = df_copy.loc[non_noisy_indices, 'food_label']

    df_copy['noisy_food_label'] = df_copy['noisy_food_label'].astype(int)

    df_shuffled = df_copy.sample(frac=1).reset_index(drop=True)

    return df_shuffled

In [ ]:
# This experiment from the article - we intentionally apply noise to label - to that good annotators 
# can help improve the labels in CEWA method even if model was trained on noisy labels 
# See results in classification_cewa.ipynb
meta_path = "/root/.cache/kagglehub/datasets/kuanghueilee/food-101n/versions/1/Food-101N_release/meta/verified_train.tsv"  # Update path
main_path = "/root/.cache/kagglehub/datasets/kuanghueilee/food-101n/versions/1/Food-101N_release/images"

df = pd.read_csv(meta_path, sep='\t', names=['image_path', 'label'])
df = df.drop(df.index[0]).reset_index(drop=True)
df = df[df.apply(check_path_exists, root_path=main_path, axis=1)].reset_index(drop=True)

clean_df = df[df['label'].notnull()]

class_names = sorted(clean_df['image_path'].str.split('/').str[0].unique())
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
clean_df['food_label'] = clean_df['image_path'].str.split('/').str[0].map(class_to_idx)
clean_df = add_noise_to_labels(clean_df, noise_ratio=0.4)

image_paths = clean_df['image_path'].tolist()
image_paths = [os.path.join(main_path, tmp_path) for tmp_path in image_paths]
labels = clean_df['noisy_food_label'].tolist()
num_classes = len(class_names)

In [ ]:
clean_df.to_csv("101n_filtered.csv", index=False)

In [ ]:
nan_rows = df[df['noisy_food_label'].isna()]
print(f"Rows with NaN values:\n{nan_rows}")

root = main_path

df['path'] = df['image_path'].apply(lambda x: os.path.join(root, str(x)))
file_exists_mask = df['path'].apply(os.path.exists)

df_existing = df[file_exists_mask].copy()
df_existing = df_existing.reset_index(drop=True)

len(df_existing), len (df)

## Prepare model

In [ ]:
class MetricTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.loss = 0
        self.preds = []
        self.targets = []

    def update(self, loss, preds, targets):
        self.loss += loss.item()
        self.preds.extend(preds.cpu().numpy())
        self.targets.extend(targets.cpu().numpy())

    def compute(self):
        metrics = {
            'loss': self.loss / len(self.targets),
            'accuracy': accuracy_score(self.targets, self.preds),
            'precision': precision_score(self.targets, self.preds, average='macro', zero_division=0),
            'recall': recall_score(self.targets, self.preds, average='macro', zero_division=0),
            'f1': f1_score(self.targets, self.preds, average='macro', zero_division=0)
        }
        self.reset()
        return metrics

class Food101NDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
feature_extractor = AutoImageProcessor.from_pretrained("microsoft/swin-tiny-patch4-window7-224")
train_transform = T.Compose([
    T.RandomResizedCrop(224),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std),
])
val_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=feature_extractor.image_mean, std=feature_extractor.image_std),
])

## Train model and get probabilities

In [ ]:
# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
all_preds = np.zeros((len(image_paths), num_classes), dtype=float)
fold_metrics = []

epochs = 1
lr = 5e-5
batch_size = 32

for fold, (train_idx, val_idx) in enumerate(kf.split(image_paths)):
    print(f"\n=== Fold {fold + 1}/5 ===")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SwinForImageClassification.from_pretrained(
        "microsoft/swin-tiny-patch4-window7-224",
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    ).to(device)

    train_dataset = Food101NDataset(
        [image_paths[i] for i in train_idx],
        [labels[i] for i in train_idx],
        transform=train_transform
    )
    val_dataset = Food101NDataset(
        [image_paths[i] for i in val_idx],
        [labels[i] for i in val_idx],
        transform=val_transform
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)

    train_metrics = MetricTracker()
    val_metrics = MetricTracker()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    model.train()

    for epoch in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
            inputs, labels_batch = batch
            inputs = inputs.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model(inputs, labels=labels_batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            preds = outputs.logits.argmax(-1)
            train_metrics.update(loss, preds, labels_batch)

        metrics = train_metrics.compute()
        print(f"\nTrain - Loss: {metrics['loss']:.4f}, "
              f"Accuracy: {metrics['accuracy']:.4f}, "
              f"Precision: {metrics['precision']:.4f}, "
              f"Recall: {metrics['recall']:.4f}, "
              f"F1: {metrics['f1']:.4f}")

        model.eval()
        val_preds = []
        with torch.no_grad():
            for batch in val_loader:
                inputs, labels_batch = batch
                inputs = inputs.to(device)
                outputs = model(inputs, labels=labels_batch)

                loss = outputs.loss  
                preds = outputs.logits.argmax(-1)
                val_metrics.update(loss, preds, labels_batch)

                val_preds.extend(torch.softmax(outputs.logits, dim=1).cpu().numpy())

        all_preds[val_idx] = val_preds

        metrics = val_metrics.compute()
        fold_metrics.append(metrics)
        print(f"\nValidation - Loss: {metrics['loss']:.4f}, "
              f"Accuracy: {metrics['accuracy']:.4f}, "
              f"Precision: {metrics['precision']:.4f}, "
              f"Recall: {metrics['recall']:.4f}, "
              f"F1: {metrics['f1']:.4f}")

## Save results

In [ ]:
np.save("data/swin_food101n_predictions.npy", all_preds)
pd.DataFrame(all_preds).to_csv("data/swin_food101n_probabilities.csv", index=False)
clean_df.to_csv("data/clean_101n_filtered_with_noisy_column.csv", index=False)